# Autoship Nudge Promo Incentive — Power Analysis (Subsequent Fix Order Rate)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Subsequent Fix Order Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes the experiment for its primary metric, **Subsequent Fix Order Rate**.

## Population
Manual clients — i.e., not already enrolled in Autoship — who completed First Fix checkout with a **Buy 1+** keep rate (kept at least one item). Buy 0 clients always see the BAU Quick Fix experience with no Autoship nudge and are out of scope for this comparison, per the PRD.

## Design: 3-cell test, two planned comparisons
Eligible clients are randomized into 3 cells at a 33/33/33 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment 1 | Autoship nudge, no promo | None |
| Treatment 2 | Autoship nudge + promo billboard | 10% off next eligible Fix |

Two pairwise comparisons are planned, matching the goal each treatment cell is designed to answer:
1. **Treatment 1 vs. Control** — establishes BAU non-promo nudge performance.
2. **Treatment 2 vs. Treatment 1** — measures the incremental lift the promo incentive adds on top of the non-promo nudge. This is the comparison the rollout decision hinges on.

Because two comparisons are planned against the same family-wise error budget, this notebook sizes using a **Bonferroni-adjusted alpha**: `0.05 / 2 = 0.025` per comparison. The sizing below is run for comparison 2 (Treatment 2 vs. Treatment 1), since that is the comparison the promo-vs-non-promo rollout decision is based on; Treatment 1 vs. Control replicates an already-approved effect and is not separately sized here.

## One-sided test
For Subsequent Fix Order Rate, a **flat** result (no incremental lift in subsequent Fix orders from the promo) and a **negative** result (the promo nudge leads to fewer subsequent Fix orders than the non-promo nudge) lead to the identical rollout decision: continue with the non-promo nudge as BAU. There is no decision on the table that requires distinguishing "no incremental effect" from "a harmful incremental effect" of the promo — so this sizing is **one-sided**, powered only to detect a positive lift from adding the promo incentive on top of the non-promo nudge.

## Multiple-comparison method: why Bonferroni
**Bonferroni** controls the family-wise Type I error rate; the cost of that control is reduced power relative to more powerful alternatives. **Holm-Bonferroni** achieves the same FWER control with uniformly more power, but it steps through ordered p-values at analysis time — pre-experiment, which comparison will rank where is unknown, so the only defensible planning assumption is the worst case, which is Bonferroni's flat `alpha / m`. **Dunnett's test** would be more powerful still for a shared-control design, but it has no closed-form sample-size formula; pre-experiment power under Dunnett requires simulation, not a general-purpose planning calculation. Bonferroni is used here because it is the only one of the three with a closed-form answer at the planning stage.

## Metric definition
**Subsequent Fix Order Rate** = share of eligible clients who complete another Fix checkout within a 90-day window following their First Fix checkout.

This notebook reads order status directly from `curated.merch_sales_and_feedback`, an item-level Fix/direct-buy fact table: a client is counted as having **ordered a subsequent Fix** if they have any shipment tagged `fix_number >= 2` whose `checkout_date` falls within 90 days of their First Fix's `checkout_date`.

This is a **downstream, fulfillment-based** definition: it only counts an order once it has actually been checked out (paid for), not when the client first decides to schedule one. That has one known blind spot worth flagging: a client who schedules a subsequent Fix but whose Fix hasn't shipped yet, or gets cancelled, before the 90-day window closes is counted as "did not order" here, even though they did take the ordering action this experiment is trying to measure. That undercount is directionally conservative for sizing purposes (it can only push the baseline rate used here down, not up), but it means the true order rate is likely somewhat higher than this notebook's baseline.

The advantage of this definition is that it requires no assumptions about intermediate demand-tracking fields — a completed Fix checkout is an unambiguous, fully-resolved outcome, sourced from the same fact table already used elsewhere in this experiment's eligibility logic.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
MATURATION_DAYS = 90  # days to wait for a fresh post-First-Fix subsequent-order signal to resolve

# Design parameters (3-cell test, Bonferroni-adjusted alpha for 2 planned comparisons)
INITIAL_ALPHA = 0.05
N_COMPARISONS = 2  # Treatment 1 vs Control, Treatment 2 vs Treatment 1
ALPHA = INITIAL_ALPHA / N_COMPARISONS  # = 0.025 Bonferroni
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive incremental lift from the promo changes the rollout decision
N_ARMS = 3
SPLIT = 0.5  # Treatment 1 and Treatment 2 are equal-sized arms (each 1/3 of the 33/33/33 split), so the pairwise comparison between them is a 50/50 split
MDE_GRID = [0.02, 0.03, 0.04, 0.05]  # relative lift on Subsequent Fix Order Rate, promo vs. non-promo

## Step 1 — Subsequent Fix Order Rate baseline & daily eligible volume

The eligible population is identified from `curated.merch_sales_and_feedback`: each client's earliest `fix_number = 1` shipment, gated to Manual (`autoship_or_manual = 'manual'`) + Buy 1+ (at least one item kept), with a 90-day maturation cutoff so the subsequent-order read has had time to resolve. A client's order status is then read from the same table: any later shipment tagged `fix_number >= 2` whose checkout falls within the 90-day window.

In [2]:
baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
next_fix AS (
    -- Each client's earliest Fix checkout at fix_number 2 or later, regardless of how it was
    -- fulfilled (manual or Autoship) -- this metric only cares whether another Fix was ordered.
    SELECT client_id, MIN(checkout_date) AS next_fix_checkout_date
    FROM curated.merch_sales_and_feedback
    WHERE fix_number >= 2
    GROUP BY client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN n.next_fix_checkout_date IS NOT NULL
              AND n.next_fix_checkout_date <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS ordered_subsequent_fix
    FROM eligible e
    LEFT JOIN next_fix n ON n.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(ordered_subsequent_fix) AS n_ordered,
    CAST(SUM(ordered_subsequent_fix) AS DOUBLE) / COUNT(*) AS subsequent_fix_order_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_ordered,subsequent_fix_order_rate,eligible_per_day
0,2026-05-01,8,2678,1273,0.475355,334.8
1,2026-04-01,30,10260,4775,0.465400,342.0
2,2026-03-01,31,10857,4880,0.449480,350.2
3,2026-02-01,28,8800,4181,0.475114,314.3
4,2026-01-01,31,10366,4753,0.458518,334.4
5,2025-12-01,31,8917,4294,0.481552,287.6
6,2025-11-01,30,7250,3156,0.435310,241.7
7,2025-10-01,31,9202,3867,0.420235,296.8
8,2025-09-01,30,9272,3838,0.413934,309.1
9,2025-08-01,31,7167,3005,0.419283,231.2


In [3]:
# Reference month = most recent calendar month fully past the 90-day maturation cutoff as of this run.
REFERENCE_MONTH = '2026-04-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['subsequent_fix_order_rate'][0])

print(f"BASELINE_RATE = {BASELINE_RATE:.4f}")
ref.T

BASELINE_RATE = 0.4654


,0
month,2026-04-01
days_observed,30
n_eligible,10260
n_ordered,4775
subsequent_fix_order_rate,0.4654
eligible_per_day,342.0


**Reference month choice:** the most recent month with every calendar day already past the 90-day maturation cutoff — more recent months are only partially mature, so their rates aren't yet comparable to a full month. This is used for **BASELINE_RATE** only; see Step 1b for why eligible **volume** is read from a different, more recent month instead.

## Step 1b — A fresher volume read (decoupled from the maturation gate)

The eligible population (Manual, Buy 1+, First-Fix-complete) has grown substantially in recent months as clients shift away from pre-checkout Autoship enrollment toward the post-checkout nudge flow this experiment targets — meaning a stale reference month understates the population's current daily run rate by a wide margin. Unlike the order-rate read, eligible volume doesn't need the 90-day maturation wait (Manual + Buy 1+ status is known immediately at First Fix checkout), so it can be measured off the **most recent complete month** instead of the same reference month as the rate.

In [4]:
volume_query = """--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
"""

volume_df = query(volume_query)
volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-08-01,6,3318,553.0
1,2026-07-01,31,17747,572.5
2,2026-06-01,30,12149,405.0
3,2026-05-01,31,6821,220.0
4,2026-04-01,1,10,10.0


In [5]:
# Volume reference month = most recent *complete* calendar month (no maturation gate needed for volume).
VOLUME_REFERENCE_MONTH = '2026-07-01'
vol_ref = volume_df[volume_df['month'].astype(str).str.startswith(VOLUME_REFERENCE_MONTH)].reset_index(drop=True)

DAILY_ELIGIBLE = float(vol_ref['eligible_per_day'][0])

print(f"BASELINE_RATE (Apr 2026, matured) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE (Jul 2026, current run rate) = {DAILY_ELIGIBLE:,.1f}")

BASELINE_RATE (Apr 2026, matured) = 0.4654  |  DAILY_ELIGIBLE (Jul 2026, current run rate) = 572.5


**Reading this:** eligible volume grows substantially between April and July 2026 as the Manual + Buy 1+ First-Fix population becomes a larger share of all First Fixes. Step 2 and Step 3 below use July volume for duration math, paired with April's matured rate for the proportions math — the two are measured over different windows deliberately, since each is gated by a different constraint (rate needs 90-day maturation; volume does not).

## Step 2 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment 2 vs. Treatment 1); `n_treatment` is read as the **per-arm** requirement. Each of the 3 arms accrues `DAILY_ELIGIBLE / 3` eligible clients per day under the 33/33/33 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE / 3)`.

In [6]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_3arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_3arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Subsequent Fix Order Rate, Treatment 2 vs. Treatment 1 (baseline={BASELINE_RATE:.1%}, alpha={ALPHA} Bonferroni, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- Subsequent Fix Order Rate, Treatment 2 vs. Treatment 1 (baseline=46.5%, alpha=0.025 Bonferroni, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_3arm,days_required,weeks_required
0,+2%,0.474708,45133,135399,237,33.9
1,+3%,0.479362,20070,60210,106,15.1
2,+4%,0.484016,11294,33882,60,8.6
3,+5%,0.48867,7231,21693,38,5.4


**Reading this:** at single-digit relative MDEs, required runtime is long. This is a direct consequence of sizing for a fixed *relative* lift: for a lower baseline, the same relative lift is a smaller absolute lift, which takes longer to distinguish from noise at 3 arms' worth of traffic-splitting. If the true incremental effect of the promo is in the low single digits, this design implies a longer runway than a larger MDE would need. No harm/guardrail grid is computed here: sizing for a one-sided positive MDE does not symmetrically size for detecting harm, and downside risk on this metric is out of scope for this sizing exercise (margin risk is covered qualitatively via the Experiment Design doc's Risk section and Decision Matrix, not powered here).

## Step 3 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 4% relative lift** on Subsequent Fix Order Rate (Treatment 2 vs. Treatment 1) — the second-largest value in the Step 2 grid. Every value in this grid runs several months or longer (Step 2).

In [7]:
TARGET_REL_MDE = 0.04  # placeholder: 4% relative lift on Subsequent Fix Order Rate, promo vs. non-promo (second-largest value in the Step 2 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Subsequent Fix Order Rate (Treatment 2 promo vs. Treatment 1 non-promo)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship',
    'Baseline Value': f"{BASELINE_RATE:.1%} ({REFERENCE_MONTH[:7]}, {MATURATION_DAYS}-day matured cohort)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.0f} / day ({VOLUME_REFERENCE_MONTH[:7]}, current run rate)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{INITIAL_ALPHA} initial alpha; {ALPHA} per comparison (Bonferroni, m={N_COMPARISONS})",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '33% / 33% / 33% (Control / Treatment 1 / Treatment 2)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (3 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Subsequent Fix Order Rate (Treatment 2 promo vs. Treatment 1 non-promo)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship"
Baseline Value,"46.5% (2026-04, 90-day matured cohort)"
Daily Eligible Volume,"572 / day (2026-07, current run rate)"
Minimum Detectable Effect,+4% relative (0.465 -> 0.484)
One/Two-Sided Test,One-sided
Significance Level,"0.05 initial alpha; 0.025 per comparison (Bonferroni, m=2)"
Statistical Power,80%
Variant Split %,33% / 33% / 33% (Control / Treatment 1 / Treatment 2)
Minimum Samples by Variant,"11,294"


## Bottom line

- **Subsequent Fix Order Rate baseline**, read as a completed second-or-later Fix checkout within 90 days of First Fix, is a simple, fully-resolved definition sourced from the same fact table already used for this experiment's eligibility logic.
- **This definition understates the true order rate** for clients who schedule a subsequent Fix that hasn't shipped yet, or that gets cancelled, within the 90-day window — a directionally conservative bias for sizing purposes.
- **Eligible daily volume has grown substantially** as the population shifts toward the post-checkout nudge flow (Step 1b), so this notebook sizes using a matured rate paired with a fresher volume read rather than a single stale reference month.
- **The primary decision-relevant comparison is Treatment 2 vs. Treatment 1** (promo vs. non-promo), sized one-sided at a Bonferroni-adjusted `alpha = 0.025` (2 planned comparisons in this 3-cell design).
- At a **4% relative lift** (the placeholder MDE above), the experiment needs the sample size and duration shown in Step 3. Smaller MDEs (2-3%) push runtime out considerably further, while 5% is comparatively faster (Step 2).